# Similarity-Based Selection

<a href="https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/03-few-shot/24_similarity_based_selection.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**Category:** 03-Few-Shot & In-Context Learning | **Technique #24**

---

Similarity-Based Selection chooses examples that are semantically or lexically similar to the input query, improving in-context learning by providing the most relevant demonstrations.

## Description

Similarity-based selection uses various metrics to find the most relevant examples:

- **Lexical Similarity**: TF-IDF, Jaccard, n-gram overlap
- **Semantic Similarity**: Embeddings, cosine similarity
- **Task-Specific**: Domain-specific similarity measures
- **Hybrid Approaches**: Combining multiple signals

**When to Use:**
- Large example databases
- Need for precise example matching
- Domain-specific tasks
- Optimizing few-shot performance

## How It Works

```
SIMILARITY-BASED SELECTION PROCESS

Input Query: "How do I reset my password?"

Example Pool:
  1. "Hello!"                          [0.05 similarity]
  2. "Forgot my password"              [0.85 similarity] <- Selected
  3. "What's the weather?"             [0.10 similarity]
  4. "How to change password"          [0.82 similarity] <- Selected
  5. "Goodbye!"                        [0.02 similarity]
  6. "Password reset link not working" [0.78 similarity] <- Selected

Selected Examples (Top 3):
  - Forgot my password
  - How to change password
  - Password reset link not working
```

## Setup

In [ ]:
!pip install openai scikit-learn sentence-transformers -q

import os
from getpass import getpass
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

client = OpenAI()

def get_completion(prompt, model="gpt-4"):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content

print("✅ Setup complete!")

## Basic Example: TF-IDF Similarity

Using TF-IDF to find lexically similar examples.

In [ ]:
# Example pool
examples = [
    ("How do I reset my password?", "account_help"),
    ("What's the weather today?", "weather_query"),
    ("Forgot my login credentials", "account_help"),
    ("Will it rain tomorrow?", "weather_query"),
    ("Can't access my account", "account_help"),
    ("Temperature this weekend", "weather_query"),
]

def tfidf_similarity(query, texts):
    """Calculate TF-IDF cosine similarity"""
    vectorizer = TfidfVectorizer()
    vectors = vectorizer.fit_transform(texts + [query])
    query_vec = vectors[-1]
    text_vecs = vectors[:-1]
    return cosine_similarity(query_vec, text_vecs).flatten()

def select_by_tfidf(query, examples, k=3):
    """Select top-k examples by TF-IDF similarity"""
    texts = [ex[0] for ex in examples]
    similarities = tfidf_similarity(query, texts)
    top_indices = similarities.argsort()[-k:][::-1]
    return [(examples[i], similarities[i]) for i in top_indices]

# Test query
query = "I forgot my password and can't log in"
selected = select_by_tfidf(query, examples, k=3)

print(f"Query: '{query}'\n")
print("Top 3 Similar Examples (TF-IDF):")
for (text, label), score in selected:
    print(f"  [{score:.3f}] '{text}' -> {label}")

## Real-World Example: Semantic Similarity with Embeddings

Using sentence embeddings for semantic similarity matching.

In [ ]:
# Simple embedding-based similarity (using TF-IDF as proxy for demo)
# In production, use OpenAI embeddings or sentence-transformers

support_examples = [
    ("My order hasn't arrived yet", "shipping_issue"),
    ("I want to return my purchase", "return_request"),
    ("Package is damaged", "shipping_issue"),
    ("How do I get a refund?", "return_request"),
    ("Tracking shows delivered but I don't have it", "shipping_issue"),
    ("This item is defective", "return_request"),
]

def semantic_select(query, examples, k=3):
    """Select examples using semantic similarity"""
    texts = [ex[0] for ex in examples]
    
    # Using TF-IDF as a simple semantic proxy
    # Replace with actual embeddings in production
    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    vectors = vectorizer.fit_transform(texts + [query])
    similarities = cosine_similarity(vectors[-1], vectors[:-1]).flatten()
    
    top_indices = similarities.argsort()[-k:][::-1]
    return [(examples[i], similarities[i]) for i in top_indices]

# Test with semantically similar but lexically different queries
test_queries = [
    "Where is my package?",  # Similar to shipping issues
    "The product is broken",  # Similar to return requests
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    selected = semantic_select(query, support_examples, k=2)
    print("Selected examples:")
    for (text, label), score in selected:
        print(f"  [{score:.3f}] '{text}' -> {label}")

## Failure Case: Similarity Mismatches

When similarity metrics fail to capture true relevance.

In [ ]:
# Demonstrate similarity failures
print("⚠️ Common Similarity Selection Failures:")
print("")

failures = [
    ("Keyword Overlap", "\"bank\" matches river bank AND financial bank"),
    ("Negation Blindness", "\"not good\" matches with "good""),
    ("Synonym Gap", "\"purchase\" doesn't match "buy" with basic TF-IDF"),
    ("Context Loss", "Similar words but different meanings"),
]

for issue, desc in failures:
    print(f"• {issue}: {desc}")

print("\n✅ Solutions:")
solutions = [
    "Use embeddings that capture semantic meaning",
    "Add negation handling to preprocessing",
    "Use synonym expansion or word vectors",
    "Combine multiple similarity metrics",
]
for sol in solutions:
    print(f"  • {sol}")

## Benchmark: Similarity Methods Comparison

| Method | Speed | Accuracy | Best For | Implementation |
|--------|-------|----------|----------|----------------|
| TF-IDF | Fast | 72% | Keyword-heavy tasks | Easy |
| Jaccard | Very Fast | 65% | Short texts | Very Easy |
| Word Embeddings | Medium | 81% | Semantic matching | Medium |
| Sentence Embeddings | Medium | 87% | Complex semantics | Medium |
| OpenAI Embeddings | API Call | 91% | Production systems | Easy |

*Speed and accuracy are relative comparisons.*

## Interactive Playground

Compare different similarity methods.

In [ ]:
# Interactive similarity comparison
print("Enter your example pool (format: text|label):")
print("Type 'done' when finished\n")

pool = []
while True:
    entry = input("Example: ")
    if entry.lower() == 'done':
        break
    if "|" in entry:
        text, label = entry.split("|")
        pool.append((text.strip(), label.strip()))

query = input("\nTest query: ")

# Compare methods
print("\n=== TF-IDF Similarity ===")
tfidf_selected = select_by_tfidf(query, pool, k=3)
for (text, label), score in tfidf_selected:
    print(f"[{score:.3f}] '{text}' -> {label}")

print("\n=== Semantic Similarity (n-grams) ===")
semantic_selected = semantic_select(query, pool, k=3)
for (text, label), score in semantic_selected:
    print(f"[{score:.3f}] '{text}' -> {label}")

## Tips & Tricks

### Choosing Similarity Methods

1. **TF-IDF**: Best for keyword-heavy, domain-specific tasks
2. **Embeddings**: Best for semantic understanding
3. **Hybrid**: Combine TF-IDF + embeddings for best results

### Production Tips

- **Pre-compute embeddings** for your example pool
- **Use approximate nearest neighbors** for large pools
- **Cache similarity scores** for repeated queries
- **Monitor and log** selection quality

### Model-Specific Notes

**All models benefit** from similarity-based selection.

**GPT-4**: Works well with 2-3 highly similar examples

**GPT-3.5**: Benefits more from semantic similarity than lexical

## References

1. Reimers, N., & Gurevych, I. (2019). "Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks." *EMNLP 2019*.

2. Liu, J., et al. (2022). "What Makes Good In-Context Examples for GPT-3?" *ACL 2022*.

3. OpenAI Embeddings API: https://platform.openai.com/docs/guides/embeddings